# 1. Introducción

**Problema industrial:** Organización de activos y sensores en PI Asset Framework.

**Activo analizado:** Jerarquía Planta > Área Molienda > PUMP101 > Sensores.

**Origen de datos:** Tags asociados a rutas de activo en PI AF (simulado).

**Objetivo del análisis:** Agrupar mediciones por componente y construir dashboard multi-tag.


# 2. Carga de librerías

In [ ]:
import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

LAB_DIR = Path.cwd()
os.chdir(LAB_DIR)
OUTPUT_DIR = LAB_DIR / "outputs"
OUTPUT_DIR.mkdir(exist_ok=True)
EXCEL_DIR = LAB_DIR / "excel"
DATA_PATH = LAB_DIR / "data" / "datos_exportados_PI.csv"


# 3. Lectura de datos PI System

Simulamos una exportación del historiador PI con columnas: `Timestamp`, `Tag`, `Value`, `Unit`, `Quality`.

In [ ]:
df_pi = pd.read_csv(DATA_PATH, parse_dates=["Timestamp"])
print(f"Registros cargados: {len(df_pi):,}")
df_pi.head(10)


# 4. Exploración del dato

In [ ]:
print("Columnas:", df_pi.columns.tolist())
print("\nEstadísticas por tag:")
display(df_pi.groupby("Tag")["Value"].describe())

calidad = df_pi["Quality"].value_counts(normalize=True) * 100
print("\nCalidad del dato (%):")
print(calidad.round(2))

faltantes = df_pi["Value"].isna().sum()
print(f"\nValores faltantes: {faltantes}")

df_good = df_pi[df_pi["Quality"] == "GOOD"].copy()
tendencia = df_good.groupby("Tag")["Value"].agg(["mean", "std", "min", "max"])
print("\nTendencia central por tag:")
display(tendencia)


# 5. Análisis matemático

Mapeo tag → componente usando metadatos del modelo de ingeniería.

In [ ]:
tags_meta = pd.read_excel(EXCEL_DIR / "modelo_ingenieria.xlsx", sheet_name="Tags")
df_join = df_good.merge(tags_meta, on="Tag", how="left")

resumen_componente = (
    df_join.groupby(["AssetPath", "Componente"])["Value"]
    .agg(["count", "mean", "std"])
    .reset_index()
    .rename(columns={"count": "N_muestras", "mean": "Promedio", "std": "Desv_Est"})
)
resultados_export = resumen_componente
display(resultados_export)


# 6. Visualizaciones

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.ravel()

for ax, (tag, grupo) in zip(axes, df_join.groupby("Tag"), strict=False):
    g = grupo.set_index("Timestamp")["Value"].resample("6h").mean()
    ax.plot(g.index, g.values)
    ax.set_title(tag)
    ax.grid(True, alpha=0.3)

plt.suptitle("Dashboard multi-tag por componente — PUMP101")


# 7. Exportación

In [ ]:
resultados_path = OUTPUT_DIR / "resultado_analisis.csv"
graficos_path = OUTPUT_DIR / "graficos.png"
excel_resultado = EXCEL_DIR / "modelo_resultado.xlsx"

resultados_export.to_csv(resultados_path, index=False)
with pd.ExcelWriter(excel_resultado, engine="openpyxl") as writer:
    resultados_export.to_excel(writer, sheet_name="Por_Componente", index=False)
    tags_meta.to_excel(writer, sheet_name="Tags", index=False)

plt.tight_layout()
plt.savefig(graficos_path, dpi=150, bbox_inches="tight")
print(f"CSV exportado: {resultados_path}")
print(f"Gráficos exportados: {graficos_path}")
print(f"Excel exportado: {excel_resultado}")


# 8. Interpretación ingenieril

## Interpretación para mantenimiento

La estructura AF permite priorizar mantenimiento por componente. Rodamiento y proceso muestran variabilidad distinta; integrar alertas por componente mejora la trazabilidad de intervenciones.
